# Phase 2 (L-NER) - Post-processing and ensembling of saved predictions

No training, no GPU. Reads the per-run outputs of `phase2_finetune_kaggle.ipynb` (`<run>/val_predictions.json`
and `<run>/prediction.json`) from the attached results bundles and builds new predictions from them:

* **`vote`** (document-level label consistency): within one document, every occurrence of the same string
  gets the label it was predicted with most often in that document (e.g. a name tagged A.COUNSEL twice and
  R.COUNSEL once becomes A.COUNSEL everywhere).
* **`propagate`** (document-level recall): a string predicted as an entity somewhere in a document is also
  tagged at its other exact occurrences in that document (whole-word matches that do not overlap a prediction).
* **ensemble**: span-level majority vote across runs - a span (start, end, label) is kept if at least
  `VOTE_MIN` of the runs predicted it; overlapping survivors are resolved by more votes, then longer span.

Every combination is scored on the validation documents (strict macro-F1, same scoring as the evaluator) and
written as its own `submission.zip`. Note: each run's validation set was also used to pick its best epoch, so
all validation scores here are slightly optimistic - by the same mechanism for every candidate.

**Inputs:** the HF secret `HF` (for the document texts and validation gold) and the results bundles as datasets.
**Output:** `/kaggle/working/phase2_pp_bundle.zip` (log, table, every candidate's submission.zip).

In [ ]:
# ============================ CONFIG ============================
import os
RESULTS_GLOB = ("/kaggle/input/**/val_predictions.json" if os.path.isdir("/kaggle/input")   # attached bundles
                else "phase2_results_bundle*/**/val_predictions.json")   # locally: the bundles next to this notebook
SKIP_RUNS = []             # run names to leave out entirely
ENSEMBLE_RUNS = None       # run names to ensemble (None = every run found); at least 2
VOTE_MIN = None            # ensemble: keep a span predicted by >= VOTE_MIN runs (None = strict majority)
PROP_MIN_CHARS = 3         # propagate: only strings at least this long (short strings match too much)
PROP_LABELS = None         # propagate: only these labels, e.g. ["WIT", "AUTH"] (None = all labels)
STEP_SETS = [(), ("vote",), ("propagate",), ("vote", "propagate")]   # applied in this order to each base
GREEDY = True              # also build ensembles by greedy forward selection on the validation docs (per step set)
# The values above are starting values, not tuned ones.

In [ ]:
import os, re, json, glob, zipfile
from collections import Counter, defaultdict

WORK = "/kaggle/working" if os.path.isdir("/kaggle/working") else os.path.abspath("pp_work")
os.makedirs(WORK, exist_ok=True)
_log = open(os.path.join(WORK, "phase2_pp_log.txt"), "w", encoding="utf-8")

def log(*a):
    s = " ".join(str(x) for x in a)
    print(s); _log.write(s + "\n"); _log.flush()

def load_phase2_files():
    local = os.environ.get("P2_LOCAL_DATA")                       # local testing only
    if local:
        return {f: os.path.join(local, f) for f in os.listdir(local)}
    from huggingface_hub import HfApi, hf_hub_download, get_token
    token = os.environ.get("HF_TOKEN") or get_token()     # get_token(): a token saved by `hf auth login`
    if not token:
        try:
            from kaggle_secrets import UserSecretsClient
            token = UserSecretsClient().get_secret("HF")
        except Exception as e:
            log(f"Kaggle secret 'HF' not readable ({e!r})")
    assert token, ("no Hugging Face token. Kaggle: Add-ons -> Secrets -> secret named HF, checkbox ON for this notebook. "
                   "Locally: run `hf auth login` once in a terminal (or set the HF_TOKEN environment variable)")
    repo = "Exploration-Lab/CS-781-Capstone"
    files = [f for f in HfApi().list_repo_files(repo, repo_type="dataset", token=token) if f.startswith("Phase_2_LNER/")]
    return {os.path.basename(f): hf_hub_download(repo, f, repo_type="dataset", token=token) for f in files}

P2 = load_phase2_files()
read = lambda n: json.load(open(P2[n], encoding="utf-8"))
LABELS = [l.strip() for l in open(P2["labels.txt"], encoding="utf-8") if l.strip()]
DOCS, GOLD = {}, {}
for fd in ("fold1", "fold2"):
    T, G = read(f"{fd}-text.json"), read(f"{fd}-gold.json")
    for did, text in T.items():
        DOCS[did] = text
        GOLD[did] = sorted((int(s), int(e), l) for s, e, l, *_ in G.get(did, []))
TEST = read("fold3-text.json")

def score(gold, pred, ids):
    st = {l: [0, 0, 0] for l in LABELS}
    for d in ids:
        g, p = set(gold[d]), set(pred.get(d, []))
        for l in LABELS:
            gs = {x for x in g if x[2] == l}; ps = {x for x in p if x[2] == l}
            st[l][0] += len(gs & ps); st[l][1] += len(ps - gs); st[l][2] += len(gs - ps)
    per = {}
    for l, (tp, fp, fn) in st.items():
        pr = tp / (tp + fp) if tp + fp else 0.0; rc = tp / (tp + fn) if tp + fn else 0.0
        per[l] = 2 * pr * rc / (pr + rc) if pr + rc else 0.0
    mp = sum((st[l][0] / (st[l][0] + st[l][1]) if st[l][0] + st[l][1] else 0.0) for l in LABELS) / len(LABELS)
    mr = sum((st[l][0] / (st[l][0] + st[l][2]) if st[l][0] + st[l][2] else 0.0) for l in LABELS) / len(LABELS)
    return dict(macro_f1=sum(per.values()) / len(LABELS), macro_p=mp, macro_r=mr, per_label=per)

In [ ]:
# ---- the saved predictions of every run ----
spans_of = lambda d: {k: sorted({(int(x[0]), int(x[1]), x[2]) for x in v}) for k, v in d.items()}
RUNS, TEST_ONLY = {}, {}
for f in sorted(glob.glob(RESULTS_GLOB, recursive=True)):
    folder = os.path.dirname(f); name = os.path.basename(folder)
    if "smoke" in folder.replace("\\", "/").split("/") or name in SKIP_RUNS: continue
    if not os.path.exists(os.path.join(folder, "prediction.json")): continue
    if name in RUNS:
        log(f"run {name} found twice, keeping {RUNS[name]['folder']}"); continue
    cfg = json.load(open(os.path.join(folder, "val_metrics.json"))).get("config", {}) \
        if os.path.exists(os.path.join(folder, "val_metrics.json")) else {}
    if cfg.get("train_on_val"):                   # no honest validation score: only its test submissions are built
        TEST_ONLY[name] = spans_of(json.load(open(os.path.join(folder, "prediction.json"), encoding="utf-8")))
        log(f"run {name}: trained on the validation documents -> test submissions only, no validation score"); continue
    RUNS[name] = dict(folder=folder, val=spans_of(json.load(open(f, encoding="utf-8"))),
                      test=spans_of(json.load(open(os.path.join(folder, "prediction.json"), encoding="utf-8"))))
assert RUNS, f"no run folders found under {RESULTS_GLOB} - attach the results bundles as datasets"
val_sets = {n: frozenset(r["val"]) for n, r in RUNS.items()}
VAL_IDS = sorted(frozenset.intersection(*val_sets.values()))
if len(set(val_sets.values())) > 1:
    log("WARNING: runs were validated on different documents; scoring on the common ones only")
assert VAL_IDS, "the runs share no validation document"
log(f"{len(RUNS)} runs: {list(RUNS)}")
log(f"validation docs: {len(VAL_IDS)}, test docs: {len(TEST)}")

In [ ]:
# ---- the post-processing steps ----
WORD_EDGE_L, WORD_EDGE_R = r"(?<![A-Za-z0-9])", r"(?![A-Za-z0-9])"
norm = lambda s: " ".join(s.split()).lower()

def overlaps(s, e, spans):
    return any(s < pe and ps < e for ps, pe, _ in spans)

def ensemble(preds, k, ids):
    """preds in priority order. Keep spans with >= k votes; overlaps: more votes, then the higher-priority run, then longer."""
    out = {}
    for d in ids:
        votes, first = Counter(), {}
        for i, p in enumerate(preds):
            for sp in set(p.get(d, [])):
                votes[sp] += 1; first.setdefault(sp, i)
        kept = []
        for sp in sorted((sp for sp, n in votes.items() if n >= k), key=lambda sp: (-votes[sp], first[sp], sp[0] - sp[1], sp)):
            if not overlaps(sp[0], sp[1], kept): kept.append(sp)
        out[d] = sorted(kept)
    return out

def vote(pred, texts):
    out = {}
    for d, spans in pred.items():
        t, by = texts[d], defaultdict(Counter)
        for s, e, l in spans: by[norm(t[s:e])][l] += 1
        out[d] = sorted({(s, e, by[norm(t[s:e])].most_common(1)[0][0]) for s, e, l in spans})
    return out

def propagate(pred, texts):
    out = {}
    for d, spans in pred.items():
        t, spans = texts[d], list(spans)
        by = defaultdict(Counter)
        for s, e, l in spans:
            if e - s >= PROP_MIN_CHARS and (PROP_LABELS is None or l in PROP_LABELS): by[t[s:e]][l] += 1
        for string, c in by.items():
            lab = c.most_common(1)[0][0]
            for m in re.finditer(WORD_EDGE_L + re.escape(string) + WORD_EDGE_R, t):
                if not overlaps(m.start(), m.end(), spans): spans.append((m.start(), m.end(), lab))
        out[d] = sorted(set(spans))
    return out

STEPS = {"vote": vote, "propagate": propagate}
def apply(pred, texts, steps):
    for st in steps: pred = STEPS[st](pred, texts)
    return pred

def write_submission(name, test_pred):
    folder = os.path.join(WORK, "pp", name); os.makedirs(folder, exist_ok=True)
    pj = {}
    for d in TEST:                                # every official id, valid labels, 0 <= s < e, no duplicates
        sp = sorted({(s, e, l) for s, e, l in test_pred.get(d, []) if 0 <= s < e <= len(TEST[d]) and l in LABELS})
        pj[d] = [[s, e, l, TEST[d][s:e]] for s, e, l in sp]
    with open(os.path.join(folder, "prediction.json"), "w", encoding="utf-8") as f: json.dump(pj, f, ensure_ascii=False)
    with zipfile.ZipFile(os.path.join(folder, "submission.zip"), "w", zipfile.ZIP_DEFLATED) as z:
        z.write(os.path.join(folder, "prediction.json"), "prediction.json")
    return sum(len(v) for v in pj.values())

In [ ]:
# ---- every base (each run, and the ensemble) x every step set: score on validation, write a submission ----
VAL_TEXT = {d: DOCS[d] for d in VAL_IDS}
bases = {n: ({d: r["val"].get(d, []) for d in VAL_IDS}, r["test"]) for n, r in RUNS.items()}
ens = [n for n in (ENSEMBLE_RUNS or list(RUNS)) if n in RUNS]
if len(ens) >= 2:
    k = VOTE_MIN or len(ens) // 2 + 1
    bases[f"ens{len(ens)}_k{k}"] = (ensemble([bases[n][0] for n in ens], k, VAL_IDS), ensemble([bases[n][1] for n in ens], k, list(TEST)))
    log(f"ensemble of {ens}, a span needs >= {k} votes")

rows = []
for base, (v, t) in bases.items():
    base_f1 = None
    for steps in STEP_SETS:
        name = base + "".join("+" + s for s in steps)
        m = score(GOLD, apply(v, VAL_TEXT, steps), VAL_IDS)
        n_test = write_submission(name, apply(t, TEST, steps))
        base_f1 = m["macro_f1"] if base_f1 is None else base_f1
        rows.append(dict(name=name, base=base, steps=list(steps), val_macro_f1=m["macro_f1"], val_p=m["macro_p"],
                         val_r=m["macro_r"], diff_vs_base=m["macro_f1"] - base_f1, per_label=m["per_label"], test_spans=n_test))

# ---- greedy forward selection (Caruana et al. 2004): start from the best single run, keep adding the run that raises
# the ensemble's val F1 most, stop when no run does. A span needs the votes of at least half the selected runs.
# The selection itself uses the validation docs, so its val score is more optimistic than the other rows'.
half = lambda n: (n + 1) // 2
def ens_val(names, steps):
    return score(GOLD, apply(ensemble([bases[n][0] for n in names], half(len(names)), VAL_IDS), VAL_TEXT, steps), VAL_IDS)

if GREEDY and len(RUNS) >= 2:
    pool = list(RUNS)
    for steps in STEP_SETS:
        chosen = [max(pool, key=lambda n: ens_val([n], steps)["macro_f1"])]
        best = ens_val(chosen, steps)["macro_f1"]
        while True:
            cand = [(ens_val(chosen + [n], steps)["macro_f1"], n) for n in pool if n not in chosen]
            if not cand or max(cand)[0] <= best: break
            best, n = max(cand); chosen.append(n)
        name = f"greedy{len(chosen)}" + "".join("+" + s for s in steps)
        m = ens_val(chosen, steps)
        n_test = write_submission(name, apply(ensemble([bases[n][1] for n in chosen], half(len(chosen)), list(TEST)), TEST, steps))
        rows.append(dict(name=name, base="greedy", steps=list(steps), members=chosen, val_macro_f1=m["macro_f1"],
                         val_p=m["macro_p"], val_r=m["macro_r"], diff_vs_base=m["macro_f1"] - ens_val(chosen[:1], steps)["macro_f1"],
                         per_label=m["per_label"], test_spans=n_test))
        log(f"greedy selection{' with ' + '+'.join(steps) if steps else ''}: {chosen} (needs >= {half(len(chosen))} votes)")

for name, t in TEST_ONLY.items():                 # runs trained on the validation docs: submissions, no scores
    for steps in STEP_SETS:
        cand = name + "".join("+" + s for s in steps)
        log(f"  {cand:40s} (trained on val: no validation score) test spans {write_submission(cand, apply(t, TEST, steps))}")

log(f"\n  {'candidate':40s} {'val F1':>7s} {'P':>6s} {'R':>6s} {'vs base':>8s} {'test spans':>10s}")
for r in rows:
    if not r["steps"]: log("")
    if r["base"] == "greedy" and not r["steps"]: log("  (greedy: 'vs base' = vs. its first, best single run)")
    log(f"  {r['name']:40s} {r['val_macro_f1']:7.4f} {r['val_p']:6.3f} {r['val_r']:6.3f} {r['diff_vs_base']:+8.4f} {r['test_spans']:10d}")
log("\nper-label val F1:")
log(f"  {'candidate':40s} " + " ".join(f"{l[:7]:>7s}" for l in LABELS))
for r in rows:
    log(f"  {r['name']:40s} " + " ".join(f"{r['per_label'][l]:7.3f}" for l in LABELS))

json.dump(rows, open(os.path.join(WORK, "phase2_pp_results.json"), "w"), indent=1)
_log.flush()
bundle = os.path.join(WORK, "phase2_pp_bundle.zip")
with zipfile.ZipFile(bundle, "w", zipfile.ZIP_DEFLATED) as z:
    for f in glob.glob(os.path.join(WORK, "**", "*"), recursive=True):
        if f.endswith((".txt", ".json", ".zip")) and not f.endswith("phase2_pp_bundle.zip"):
            z.write(f, os.path.relpath(f, WORK))
log(f"\nbundle: {bundle} - each candidate's submission is pp/<candidate>/submission.zip")